# 02 — Seed, Index Audit, and the `grant_records` Hypothesis

**Phase 1, stage 2.** Stage 1 (`01_api_access_map.ipynb` + `doc-api-sql-matrix.md`) established
*which* SQL each API issues. It could not establish whether any of it is slow, because the
metastore was almost empty — a sequential scan over 40 rows is free, and `EXPLAIN` on a tiny
table proves nothing.

This notebook gives the metastore enough volume for the planner's choices to become meaningful,
then measures.

**The claim under test** (from `doc-api-sql-matrix.md` §4):

> `S4` — `SELECT ... FROM grant_records WHERE grantee_id = ? AND realm_id = ? AND grantee_catalog_id = ?`
> — runs on essentially every authenticated request (186 of 1,605 statements captured). The only
> index on `grant_records` is its primary key
> `(realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)`.
> `S4` constrains positions 1, 4, 5 and leaves 2 and 3 free, so only `realm_id` is usable as a
> leading prefix — which in a single-realm deployment selects 100% of the table.

This is falsifiable. If after seeding the planner still picks an index scan, or the timing
difference is negligible, the hypothesis is wrong and we say so.

**Order of operations matters.** Seed → `ANALYZE` → `EXPLAIN` → create index → `EXPLAIN` again.
Skipping the `ANALYZE` makes every verdict garbage: `reltuples = -1` on a never-analyzed table
means the planner is guessing, and `schema_audit` will report `TOO_SMALL` for a table that
actually holds 25,000 rows.

## 0 — Configuration and guards

Endpoints are hardcoded (local cluster only); secrets come from the environment with visible
defaults, per the project convention. The host assert is a hard stop — this notebook seeds a
thousand catalogs and must never point at anything shared.

In [ ]:
import os, sys, time, json, textwrap, pathlib, datetime

REPO = pathlib.Path.cwd()
while not (REPO / "src").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

# ---- Polaris ---------------------------------------------------------------
POLARIS_URL  = "http://192.168.139.2:8181"
REALM        = "POLARIS"
ROOT_CLIENT  = "root"
ROOT_SECRET  = os.environ.get("POLARIS_ROOT_SECRET", "polaris-secret")

# ---- PostgreSQL ------------------------------------------------------------
# Direct to the PRIMARY, deliberately bypassing Pgpool: EXPLAIN must run on the
# node that holds the data, and a load-balanced read could land on a replica
# with different planner statistics.
#   kubectl -n datahub-hynix port-forward \
#     pod/benchmarks-postgresql-postgresql-ha-postgresql-0 5433:5432
PG = dict(
    host="127.0.0.1", port=5433, dbname=os.environ.get("PG_DB", "polaris"),
    user=os.environ.get("PG_USER", "polaris"),
    password=os.environ.get("PG_PASSWORD", "polaris"),
)
SCHEMA = "polaris_schema"

# ---- MinIO (only needed if create_tables=True) -----------------------------
MINIO_ENDPOINT = "http://192.168.139.2:9000"
BUCKET         = "data-catalog-bucket"

assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), \
    "refusing to seed: POLARIS_URL is not a local address"
assert PG["host"] in ("127.0.0.1", "localhost"), \
    "refusing to run: PG host is not local (use a port-forward)"

OUT = REPO / "diagnostics" / "api-sql-profile"
print("repo    :", REPO)
print("polaris :", POLARIS_URL)
print("postgres:", f"{PG['host']}:{PG['port']}/{PG['dbname']}")

In [ ]:
try:
    import psycopg2
    import psycopg2.extras
except ImportError:
    raise SystemExit("pip install psycopg2-binary   # schema_audit is written against psycopg2")

import polaris_rest, iceberg_rest, polaris_seed, schema_audit, api_trace
from polaris_rest import PolarisREST
from iceberg_rest import IcebergREST
from polaris_seed import SeedSpec, seed, teardown, verify_counts, find_strays

conn = psycopg2.connect(**PG)
conn.autocommit = True          # CREATE INDEX CONCURRENTLY cannot run in a txn
with conn.cursor() as cur:
    cur.execute("select version(), current_database(), pg_is_in_recovery()")
    v, db, standby = cur.fetchone()
print(v.split(",")[0])
print("database :", db)
print("standby  :", standby, "  <-- must be False; EXPLAIN needs the primary")
assert not standby, "connected to a standby — port-forward the primary instead"

pc = PolarisREST(POLARIS_URL, REALM)
pc.token = pc.get_token(ROOT_CLIENT, ROOT_SECRET).json()["access_token"]
ic = IcebergREST(POLARIS_URL, REALM, token=pc.token)
print("polaris  : authenticated as", ROOT_CLIENT)

## 1 — Schema drift check

Before measuring anything, establish what is actually deployed. Decision #6 flagged the concern
that the running schema may not match the version we think it is — and an index audit against
the wrong baseline is worthless.

`compare_schema` reports missing tables, unexpected tables, and missing indexes against the
upstream `schema-v2.sql` definition. `events` is expected and is *not* drift: it is created by
the `persistence-in-memory-buffer` event listener.

In [ ]:
snap = schema_audit.describe_schema(conn, schema=SCHEMA)
cmp_ = schema_audit.compare_schema(snap)

print("verdict          :", cmp_["verdict"])
print("schema version   :", cmp_["version_found"], "(expected", cmp_["version_expected"], ")")
print()
for key in ("missing_tables", "unexpected_tables", "event_listener_tables",
            "missing_indexes", "extra_indexes"):
    vals = cmp_.get(key) or []
    mark = "!!" if vals and key.startswith("missing") else "  "
    print(f"{mark} {key:24s}: {vals if vals else 'none'}")
for note in cmp_.get("notes") or []:
    print("   note:", note)

print()
print("indexes actually present:")
for ix in sorted(snap["indexes"], key=lambda r: (r["table"], r["name"])):
    kind = "PK" if ix.get("is_primary") else ("UQ" if ix.get("is_unique") else "  ")
    print(f"   {kind} {ix['table']:28s} {ix['name']:34s} {','.join(ix['columns'])}")

## 2 — Baseline, before seeding

Row counts and the current `EXPLAIN` for `S4`. On an empty metastore this will be a sequential
scan simply because that is correct for 40 rows — which is exactly why this baseline cannot
settle the question on its own. It is recorded so the after-state has something to compare to.

In [ ]:
def show_stats(title):
    """table_stats returns pg_stat_user_tables counters: n_live_tup, seq_scan,
    idx_scan, seq_scan_ratio. seq_scan_ratio is the one to watch here — it is
    the fraction of scans on that table that were sequential."""
    print(title)
    rows = schema_audit.table_stats(conn, schema=SCHEMA)
    print(f"   {'table':30s} {'live_rows':>10} {'seq_scan':>9} {'idx_scan':>9} {'seq%':>7}")
    for r in sorted(rows, key=lambda r: -(r.get("n_live_tup") or 0)):
        ratio = r.get("seq_scan_ratio")
        print(f"   {r['table']:30s} {r.get('n_live_tup',0):>10} {r.get('seq_scan',0):>9} "
              f"{r.get('idx_scan',0):>9} {('%.0f%%' % (ratio*100)) if ratio is not None else '   -':>7}")
    return {r["table"]: r for r in rows}

before_stats = show_stats("row counts BEFORE seed")

In [ ]:
# The exact statement from doc-api-sql-matrix.md, S4.
S4 = """
SELECT securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code
FROM polaris_schema.grant_records
WHERE grantee_id = %s AND realm_id = %s AND grantee_catalog_id = %s
"""

# The grantee-arm of S13, the DELETE that shares the same access path.
S13 = """
SELECT count(*) FROM polaris_schema.grant_records
WHERE ((grantee_id = %s AND grantee_catalog_id = %s)
    OR (securable_id = %s AND securable_catalog_id = %s))
  AND realm_id = %s
"""

def hottest_grantee():
    """Pick a grantee that actually owns rows, so EXPLAIN measures a real lookup."""
    with conn.cursor() as cur:
        cur.execute(f"""SELECT grantee_id, grantee_catalog_id, count(*)
                        FROM {SCHEMA}.grant_records WHERE realm_id = %s
                        GROUP BY 1,2 ORDER BY 3 DESC LIMIT 1""", (REALM,))
        r = cur.fetchone()
    return r  # (grantee_id, grantee_catalog_id, n)

def explain(sql, params, analyze=True):
    pre = "EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) " if analyze else "EXPLAIN (FORMAT JSON) "
    with conn.cursor() as cur:
        cur.execute(pre + sql, params)
        return cur.fetchone()[0][0]

def plan_line(p):
    n = p["Plan"]
    return (f"{n['Node Type']:<22} rows={n.get('Actual Rows','?'):<8} "
            f"time={p.get('Execution Time', n.get('Actual Total Time')):.3f} ms  "
            f"shared_read={n.get('Shared Read Blocks','?')} hit={n.get('Shared Hit Blocks','?')}")

g = hottest_grantee()
if g:
    gid, gcat, n = g
    print(f"hottest grantee: id={gid} catalog={gcat} rows={n}")
    print("S4 BEFORE seed :", plan_line(explain(S4, (gid, REALM, gcat))))
else:
    print("grant_records is empty — nothing to baseline yet")

## 3 — Seed the fixture

Per decision #9: 1,000 principals, each with a principal-role, a catalog, a catalog-role, and an
`owner_principal` role holding all catalog privileges.

**`create_tables=False` is the deliberate default.** It fully populates `grant_records` — roughly
25,000 rows, which is the table the hypothesis is about — while skipping the 10,000 Iceberg
tables and their MinIO objects. Rationale:

* the index question lives entirely in `grant_records`; the tables add `entities` rows that are
  already well covered by the unique constraint;
* PostgreSQL `max_connections` is 100 against a JDBC pool sized 300, so the full seed risks
  connection exhaustion — which is a different failure to debug, not the one we are studying;
* it is far faster, and the ledger makes it resumable either way.

Set `CREATE_TABLES = True` for the full fixture once the cheap run has answered the question.

The seed is **resumable**: progress is written to `seed_ledger.json` after each user, so an
interrupted run continues rather than restarting or double-creating.

In [ ]:
CREATE_TABLES = False        # <-- the one toggle; see the note above
N_USERS       = 1000

spec = SeedSpec(
    n_users=N_USERS,
    namespaces_per_catalog=2,
    tables_per_namespace=5,
    create_tables=CREATE_TABLES,
    prefix="user",
)
print("expected entity counts:", spec.expected_counts())
print("create_tables:", CREATE_TABLES)

In [ ]:
t0 = time.time()
_last = [0]
def progress(done, total):
    if done - _last[0] >= 50 or done == total:
        _last[0] = done
        el = time.time() - t0
        rate = done / el if el else 0
        eta = (total - done) / rate if rate else 0
        print(f"  {done:>5}/{total}  {el:6.1f}s elapsed  ~{eta:5.0f}s remaining", flush=True)

result = seed(
    pc, ic, spec,
    ledger_path=str(OUT / "seed_ledger.json"),
    bucket=BUCKET if CREATE_TABLES else None,
    minio_endpoint=MINIO_ENDPOINT if CREATE_TABLES else None,
    progress_every=50,
    on_progress=progress,
)
print()
print(result.summary())
if result.failed_users:
    print("FAILED users (first 10):", result.failed_users[:10])
if result.invalid_privileges:
    print("privileges rejected by this Polaris build:", result.invalid_privileges)

In [ ]:
counts = verify_counts(conn, spec, schema=SCHEMA)
for k, v in counts.items():
    print(f"   {k:34s} {v}")

## 4 — `ANALYZE` gate (mandatory)

Nothing below this cell is trustworthy until the planner has fresh statistics. A freshly-seeded
table has `reltuples = -1` — never analyzed — and the planner will treat 25,000 rows as an
unknown. Every `EXPLAIN` and every `schema_audit` verdict depends on this.

In [ ]:
stale = schema_audit.statistics_are_stale(conn, schema=SCHEMA)
print("stale before ANALYZE:", stale or "none")

schema_audit.analyze_tables(conn, schema=SCHEMA)

stale_after = schema_audit.statistics_are_stale(conn, schema=SCHEMA)
print("stale after  ANALYZE:", stale_after or "none")
assert not stale_after, f"ANALYZE did not take on: {stale_after}"

after_stats = show_stats("\nrow counts AFTER seed + ANALYZE")

## 5 — The measurement

`EXPLAIN (ANALYZE, BUFFERS)` on `S4` against a populated `grant_records`. This is the moment the
hypothesis stands or falls.

Read the node type first: `Seq Scan` on a 25k-row table to return a handful of rows is the
predicted failure. `Index Scan` or `Index Only Scan` would falsify the claim.

In [ ]:
gid, gcat, n = hottest_grantee()
print(f"grantee id={gid} catalog={gcat} owns {n} grant rows\n")

before_plan = explain(S4, (gid, REALM, gcat))
print("S4  :", plan_line(before_plan))
print()
print(json.dumps(before_plan["Plan"], indent=1)[:1200])

In [ ]:
before_s13 = explain(S13, (gid, gcat, gid, gcat, REALM))
print("S13 :", plan_line(before_s13))

### Full statement audit

Beyond the two statements under test, run every captured statement through `EXPLAIN` and let
`schema_audit` assign verdicts (`SEQ_SCAN`, `INDEX_SCAN`, `FILTER_HEAVY`, `TOO_SMALL`,
`NO_PARAMS`, `ERROR`). Writes are planned but **not** executed — `explain_statement` uses plain
`EXPLAIN` for `INSERT`/`UPDATE`/`DELETE` unless explicitly opted in, and even then wraps them in
a rolled-back savepoint.

**One gap to close first.** Polaris logs bind parameters as `?` and `api_trace.redact_params`
strips the values, so the inventory arrives without them — and `audit_statements` correctly
refuses to `EXPLAIN` a statement whose parameters it does not have, returning `NO_PARAMS`.
Without this next cell, nearly the whole audit comes back `NO_PARAMS` and tells us nothing.

The fix: read the equality columns out of each statement's `WHERE` clause in order, then sample
a **real row** from that table and take the matching column values. The plan is then measured
against parameters that actually select something, rather than invented ones that would make
every lookup look free.

In [ ]:
import re as _re

_EQ = _re.compile(r"(\w+)\s*=\s*(?:\?|%s|\$\d+)", _re.I)

_sample_cache = {}
def _sample_row(table):
    """One real row from the table, as a dict, cached."""
    if table not in _sample_cache:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            try:
                cur.execute(f"SELECT * FROM {SCHEMA}.{table} "
                            f"WHERE realm_id = %s LIMIT 1", (REALM,))
                _sample_cache[table] = cur.fetchone() or {}
            except Exception:
                _sample_cache[table] = {}
    return _sample_cache[table]

def params_for(sql, table):
    """Bind values for a statement, in placeholder order, drawn from live data.

    Returns None when any placeholder cannot be resolved — better to leave the
    statement as NO_PARAMS than to EXPLAIN it with a value that matches nothing
    and call the resulting cheap plan a pass.
    """
    if not table:
        return None
    # `normalize_sql` collapses row-wise IN lists to a literal marker, which is
    # not valid SQL and must never reach EXPLAIN. Same for anything with no
    # WHERE clause at all (INSERTs), where the placeholders are VALUES, not
    # predicates, and sampling a row would produce a meaningless plan.
    if "<" in sql or " WHERE " not in sql.upper():
        return None
    where = _re.split(r"\sWHERE\s", sql, maxsplit=1, flags=_re.I)[-1]
    cols = [c.lower() for c, in ((m.group(1),) for m in _EQ.finditer(where))]
    n_ph = sql.count("?") + sql.count("%s")
    if not cols or len(cols) != n_ph:
        return None                      # IN-lists, ORs, INSERTs — skip rather than guess
    row = _sample_row(table)
    out = []
    for c in cols:
        if c == "realm_id":
            out.append(REALM)
        elif c in row:
            out.append(row[c])
        else:
            return None
    return tuple(out)

_resolved = 0
def enrich(inv):
    global _resolved
    for r in inv:
        p = params_for(r["sql"], r.get("table"))
        if p is not None:
            r["params"] = p
            _resolved += 1
    return inv

In [ ]:
CAPTURE = None
for cand in (OUT / "capture", REPO / "capture", OUT / "capture-seeded"):
    if (cand / "polaris.log").exists():
        CAPTURE = cand; break
print("capture dir:", CAPTURE)

records = []
if CAPTURE:
    records = api_trace.parse_polaris_log((CAPTURE / "polaris.log").read_text(errors="replace"))
    print(f"parsed {len(records)} polaris records")

inventory = api_trace.statement_inventory(records) if records else []
print(f"{len(inventory)} distinct statements in the inventory")

inventory = enrich(inventory)
print(f"{_resolved}/{len(inventory)} got usable bind parameters from live data")
print("(the rest are IN-lists, ORs or INSERTs — reported as NO_PARAMS, not guessed)")
print()

audit = schema_audit.audit_statements(conn, inventory, schema=SCHEMA, analyze=True)
print(f"  {'verdict':14s} {'calls':>6} {'ms':>8} {'tbl_rows':>9}  table / sql")
for row in sorted(audit, key=lambda r: -(r.get("calls") or 0))[:25]:
    print(f"  {str(row.get('verdict')):14s} {row.get('calls',0):>6} "
          f"{(row.get('plan_ms') or 0):>8.2f} {row.get('table_rows','?'):>9}  "
          f"{str(row.get('table')):22s} {' '.join(str(row.get('sql','')).split())[:60]}")

seq = [r for r in audit if r.get("verdict") in ("SEQ_SCAN", "FILTER_HEAVY")]
print(f"\n{len(seq)} statement(s) on a non-selective path:")
for r in seq:
    print(f"   [{r['verdict']}] {r.get('table')}  rows_filtered={r.get('rows_removed_by_filter')}")
    print("      " + " ".join(str(r.get("sql","")).split())[:150])

In [ ]:
hyp = schema_audit.check_hypotheses(conn, audit, schema=SCHEMA)
for h in hyp:
    print(f"[{h['status']:13s}] {h['id']}   (severity: {h.get('severity')})")
    print(f"    table  : {h['table']}  via {h.get('source_method')}")
    print(textwrap.fill(f"    claim  : {h.get('claim','')}", 100,
                        subsequent_indent="             "))
    for e in (h.get("evidence") or []):
        print(f"    evidence: {' '.join(str(e).split())[:120]}")
    print(f"    remedy : {h.get('remedy')}")
    print()

CONFIRMED = {h["id"]: h["status"] for h in hyp}
print("statuses:", CONFIRMED)

## 6 — Create the index and re-measure

`CREATE INDEX CONCURRENTLY` so the table is never locked against writes. Note this **cannot run
inside a transaction block** — the same constraint that made `ALTER SYSTEM SET log_statement`
fail earlier when bundled into one `psql -c`. Hence `conn.autocommit = True` in cell 0.

The `INCLUDE` clause carries the three payload columns so `S4` becomes an *index-only* scan: it
selects exactly `securable_catalog_id, securable_id, grantee_catalog_id, grantee_id,
privilege_code`, all of which are then available from the index alone.

This is a **local-only** change. It is not applied to any shared cluster from this notebook, and
§7 records the exact DDL for review before it goes anywhere else.

In [ ]:
INDEX_NAME = "idx_grant_records_grantee"
INDEX_DDL = f"""
CREATE INDEX CONCURRENTLY IF NOT EXISTS {INDEX_NAME}
    ON {SCHEMA}.grant_records (realm_id, grantee_catalog_id, grantee_id)
    INCLUDE (securable_catalog_id, securable_id, privilege_code)
"""
print(INDEX_DDL)

t0 = time.time()
with conn.cursor() as cur:
    cur.execute(INDEX_DDL)
print(f"built in {time.time()-t0:.1f}s")

schema_audit.analyze_tables(conn, schema=SCHEMA, tables=["grant_records"])

with conn.cursor() as cur:
    cur.execute("""SELECT indexrelname, pg_size_pretty(pg_relation_size(indexrelid))
                   FROM pg_stat_user_indexes
                   WHERE schemaname=%s AND relname='grant_records'""", (SCHEMA,))
    for nm, sz in cur.fetchall():
        print(f"   {nm:36s} {sz}")

In [ ]:
after_plan = explain(S4, (gid, REALM, gcat))
after_s13  = explain(S13, (gid, gcat, gid, gcat, REALM))

def ms(p):  return p.get("Execution Time")
def node(p): return p["Plan"]["Node Type"]

print(f"{'':6} {'node':<24} {'exec ms':>10}")
print(f"{'S4':6} {node(before_plan):<24} {ms(before_plan):>10.3f}   BEFORE")
print(f"{'':6} {node(after_plan):<24} {ms(after_plan):>10.3f}   AFTER")
sp = ms(before_plan) / ms(after_plan) if ms(after_plan) else float('inf')
print(f"{'':6} speedup: {sp:.1f}x")
print()
print(f"{'S13':6} {node(before_s13):<24} {ms(before_s13):>10.3f}   BEFORE")
print(f"{'':6} {node(after_s13):<24} {ms(after_s13):>10.3f}   AFTER")

VERDICT = {
    "rows_in_grant_records": n,
    "s4_before": {"node": node(before_plan), "ms": ms(before_plan)},
    "s4_after":  {"node": node(after_plan),  "ms": ms(after_plan)},
    "s13_before": {"node": node(before_s13), "ms": ms(before_s13)},
    "s13_after":  {"node": node(after_s13),  "ms": ms(after_s13)},
    "hypothesis_supported": node(before_plan).startswith("Seq")
                            and not node(after_plan).startswith("Seq"),
}
print()
print("HYPOTHESIS SUPPORTED:" , VERDICT["hypothesis_supported"])

### End-to-end effect, not just plan-level

A faster plan is not the same as a faster API. Re-run a handful of real requests and compare
wall-clock, so the result is stated in terms a reader cares about.

In [ ]:
import statistics, requests

def timeit(fn, k=15):
    xs = []
    for _ in range(k):
        t = time.perf_counter(); fn(); xs.append((time.perf_counter()-t) * 1000)
    return statistics.median(xs), min(xs), max(xs)

probes = {
    "POST /iceberg/oauth/tokens": lambda: pc.get_token(ROOT_CLIENT, ROOT_SECRET),
    "GET  /mgmt/catalogs":        lambda: pc.list_catalogs(),
    "GET  /mgmt/principal-roles": lambda: pc.list_principal_roles(),
}
print(f"{'api':<30} {'median':>9} {'min':>9} {'max':>9}   (ms)")
timings = {}
for name, fn in probes.items():
    try:
        med, lo, hi = timeit(fn)
        timings[name] = med
        print(f"{name:<30} {med:>9.1f} {lo:>9.1f} {hi:>9.1f}")
    except Exception as e:
        print(f"{name:<30} FAILED: {str(e)[:50]}")

## 7 — Write the report

In [ ]:
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
doc = OUT / "doc-index-audit.md"

lines = [
    "# Index Audit — `grant_records` grantee access path", "",
    f"Generated {now} against a locally seeded fixture.", "",
    "## Fixture", "",
    f"- principals / catalogs / principal-roles / catalog-roles: **{spec.n_users}** each",
    f"- Iceberg tables created: **{'yes' if CREATE_TABLES else 'no (create_tables=False)'}**",
    f"- `grant_records` rows: **{after_stats.get('grant_records',{}).get('n_live_tup')}**",
    f"- `entities` rows: **{after_stats.get('entities',{}).get('n_live_tup')}**",
    "", "## Result", "",
    "| statement | plan before | ms before | plan after | ms after |",
    "|---|---|---:|---|---:|",
    f"| S4 (grantee lookup) | {VERDICT['s4_before']['node']} | {VERDICT['s4_before']['ms']:.3f} "
    f"| {VERDICT['s4_after']['node']} | {VERDICT['s4_after']['ms']:.3f} |",
    f"| S13 (grant delete arm) | {VERDICT['s13_before']['node']} | {VERDICT['s13_before']['ms']:.3f} "
    f"| {VERDICT['s13_after']['node']} | {VERDICT['s13_after']['ms']:.3f} |",
    "",
    f"**Hypothesis supported: {VERDICT['hypothesis_supported']}**", "",
    "## Proposed index", "", "```sql", INDEX_DDL.strip(), "```", "",
    "Still absent from upstream `schema-v3.sql`, so a version upgrade does not supply it.", "",
    "## Hypotheses", "",
    "| id | status | severity |", "|---|---|---|",
] + [f"| `{h['id']}` | **{h['status']}** | {h.get('severity')} |" for h in hyp] + [
    "", "## Schema drift", "",
    f"- verdict: **{cmp_['verdict']}** (schema version {cmp_['version_found']}, "
    f"expected {cmp_['version_expected']})",
    f"- missing tables: {cmp_.get('missing_tables') or 'none'}",
    f"- unexpected tables: {cmp_.get('unexpected_tables') or 'none'}",
    f"- event-listener tables: {cmp_.get('event_listener_tables') or 'none'} (not drift)",
    f"- missing indexes vs schema-v2: {cmp_.get('missing_indexes') or 'none'}",
    "", "## API wall-clock after the index", "",
    "| api | median ms |", "|---|---:|",
]
lines += [f"| `{k}` | {v:.1f} |" for k, v in timings.items()]
lines += ["", "## Caveats", "",
    "- Single local node, no concurrency. Real contention is not modelled here.",
    "- `create_tables=False` means `entities` is smaller than a production catalog of this width.",
    "- Wall-clock includes the full 7-statement auth prelude, so per-API deltas understate the",
    "  per-statement improvement.", ""]

doc.write_text("\n".join(lines))
print(doc)
print("\n".join(lines[:40]))

## 8 — Teardown

Not run automatically. Removes every seeded principal, role and catalog using the ledger, so it
only deletes what this notebook created — `find_strays` first, to see what would go.

In [ ]:
strays = find_strays(pc, prefix=spec.prefix)
print(f"{len(strays)} entities match prefix '{spec.prefix}'")
for s in strays[:10]:
    print("   ", s)

In [ ]:
# Uncomment to remove the fixture.
# res = teardown(pc, ledger_path=str(OUT / "seed_ledger.json"), spec=spec,
#                progress_every=50, on_progress=progress)
# print(res.summary())

# Uncomment to drop the index and restore the original schema.
# with conn.cursor() as cur:
#     cur.execute(f"DROP INDEX CONCURRENTLY IF EXISTS {SCHEMA}.{INDEX_NAME}")